# 03 — Modelo Tabular MLP
### GalaxyNet — Classificador Morfológico de Galáxias

Rede Neural Multi-Camadas (MLP) usando apenas as 15 features fotométricas e espectroscópicas tabulares. Arquitetura: Dense(256) → BN → Dropout → Dense(128) → BN → Dropout → Dense(64) → BN → Dropout → Softmax(3).

Desbalanceamento de classes tratado com `class_weight='balanced'`.

In [ ]:
import sys
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

# Localiza src/ de forma robusta
def _find_src_dir():
    current = os.path.abspath(os.getcwd())
    for _ in range(5):
        candidate = os.path.join(current, 'src')
        if os.path.isdir(candidate):
            return candidate
        current = os.path.dirname(current)
    raise RuntimeError("Diretório src/ não encontrado.")

src_path = _find_src_dir()
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import to_categorical

from preprocessing import load_preprocessed_tabular, TABULAR_FEATURES
from models import create_mlp_model
from evaluation import evaluate_galaxy_classifier

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# Caminhos
_project_root = os.path.dirname(src_path)
PROCESSED_DIR = os.path.join(_project_root, 'data', 'processed')
MODELS_DIR    = os.path.join(_project_root, 'models')
REPORTS_DIR   = os.path.join(_project_root, 'reports', 'figures')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"TensorFlow    : {tf.__version__}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"MODELS_DIR    : {MODELS_DIR}")
print(f"Features      : {len(TABULAR_FEATURES)}")

---
## 1. Carregamento dos Dados Pré-processados

In [ ]:
X_scaled, y_labels, objids, scaler = load_preprocessed_tabular(PROCESSED_DIR)

print(f"\nX_scaled : {X_scaled.shape}  dtype={X_scaled.dtype}")
print(f"y_labels : {y_labels.shape}   classes={np.unique(y_labels)}")
print(f"objids   : {objids.shape}")

print("\nDistribuição de classes:")
unique, counts = np.unique(y_labels, return_counts=True)
for cls, n in zip(unique, counts):
    print(f"  {cls:<12}: {n:>4} ({100*n/len(y_labels):.1f}%)")

---
## 2. Encoding de Labels e Split Train/Val/Test

- `LabelEncoder` converte strings → inteiros
- `to_categorical` converte inteiros → one-hot
- Split estratificado: **70% train / 15% val / 15% test**

In [ ]:
# Encode string labels → integers → one-hot
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_labels)
y_one_hot = to_categorical(y_encoded)

num_classes = y_one_hot.shape[1]
print(f"Classes: {list(label_encoder.classes_)} → {list(range(num_classes))}")
print(f"Shape one-hot: {y_one_hot.shape}")

# Split estratificado: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp, objids_train, objids_temp = train_test_split(
    X_scaled, y_one_hot, objids,
    test_size=0.30, random_state=SEED, stratify=y_encoded
)

# Separar temp em val (50%) e test (50%) → cada um fica com 15% do total
y_temp_encoded = np.argmax(y_temp, axis=1)
X_val, X_test, y_val, y_test, objids_val, objids_test = train_test_split(
    X_temp, y_temp, objids_temp,
    test_size=0.50, random_state=SEED, stratify=y_temp_encoded
)

print(f"\nTrain : {X_train.shape[0]} amostras ({100*X_train.shape[0]/len(X_scaled):.0f}%)")
print(f"Val   : {X_val.shape[0]} amostras ({100*X_val.shape[0]/len(X_scaled):.0f}%)")
print(f"Test  : {X_test.shape[0]} amostras ({100*X_test.shape[0]/len(X_scaled):.0f}%)")

# Distribuição por split
for name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    labels = label_encoder.inverse_transform(np.argmax(y_split, axis=1))
    unique, counts = np.unique(labels, return_counts=True)
    dist = ', '.join(f"{c}:{n}" for c, n in zip(unique, counts))
    print(f"  {name}: {dist}")

---
## 3. Class Weights para Desbalanceamento

Com apenas 4 galáxias Irregular e 17 Spiral contra 135 Elliptical, usamos `class_weight='balanced'` para penalizar erros nas classes minoritárias proporcionalmente à sua escassez.

In [ ]:
# Calcular class weights balanceados a partir do conjunto de treino
y_train_integers = np.argmax(y_train, axis=1)
class_weights_array = compute_class_weight(
    'balanced',
    classes=np.unique(y_train_integers),
    y=y_train_integers
)
class_weight_dict = dict(enumerate(class_weights_array))

print("Class weights (balanced):")
for idx, weight in class_weight_dict.items():
    cls_name = label_encoder.inverse_transform([idx])[0]
    n_train = (y_train_integers == idx).sum()
    print(f"  {cls_name:<12} (idx={idx}): weight={weight:.3f}  (n_train={n_train})")

---
## 4. Criação do Modelo MLP

In [ ]:
input_shape = X_train.shape[1]
mlp_model = create_mlp_model(input_shape, num_classes)
mlp_model.summary()

---
## 5. Treinamento

- **EarlyStopping**: para o treino se `val_loss` não melhorar por 10 épocas; restaura os melhores pesos
- **ReduceLROnPlateau**: reduz o learning rate pela metade se `val_loss` estagnar por 5 épocas

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-7, verbose=1
    ),
]

history = mlp_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1,
)

print(f"\nTreinamento finalizado em {len(history.history['loss'])} épocas.")

---
## 6. Curvas de Treinamento (Loss e Accuracy)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss — MLP', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (Categorical Crossentropy)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy — MLP', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'fig16_mlp_training_curves.png'),
            bbox_inches='tight')
plt.show()
print('Figura salva: fig16_mlp_training_curves.png')

---
## 7. Avaliação no Conjunto de Teste

Métricas: Classification Report, Confusion Matrix, Completeness (Recall) e Reliability (Precision) por classe — métricas padrão em astronomia para avaliação de classificadores de surveys.

In [ ]:
# Avaliação completa no conjunto de teste
loss, accuracy = mlp_model.evaluate(X_test, y_test, verbose=0)
print(f"MLP Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

eval_results = evaluate_galaxy_classifier(
    model=mlp_model,
    X_test_data=X_test,
    y_test_one_hot=y_test,
    label_encoder=label_encoder,
    model_type='MLP',
    save_dir=REPORTS_DIR,
)

---
## 8. Salvar Modelo e LabelEncoder

In [ ]:
# Salvar modelo treinado
model_path = os.path.join(MODELS_DIR, 'mlp_galaxy_classifier.h5')
mlp_model.save(model_path)
print(f"Modelo salvo em: {model_path}")

# Salvar LabelEncoder para reutilização nos notebooks 04 e 05
le_path = os.path.join(MODELS_DIR, 'label_encoder.pkl')
with open(le_path, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f"LabelEncoder salvo em: {le_path}")

# Salvar dados de teste para comparação posterior entre modelos
test_artifacts = {
    'X_test': X_test,
    'y_test': y_test,
    'objids_test': objids_test,
}
test_path = os.path.join(MODELS_DIR, 'mlp_test_data.pkl')
with open(test_path, 'wb') as f:
    pickle.dump(test_artifacts, f)
print(f"Dados de teste salvos em: {test_path}")

---
## 9. Resumo

In [ ]:
print("=" * 55)
print("RESUMO — MODELO MLP TABULAR")
print("=" * 55)
print(f"Features de entrada  : {input_shape}")
print(f"Classes              : {list(label_encoder.classes_)}")
print(f"Amostras (train/val/test): {X_train.shape[0]}/{X_val.shape[0]}/{X_test.shape[0]}")
print(f"Épocas treinadas     : {len(history.history['loss'])}")
print(f"Test Loss            : {loss:.4f}")
print(f"Test Accuracy        : {accuracy:.4f}")
print()
print("Métricas Científicas:")
print(eval_results['scientific_metrics'].to_string(index=False))
print()
print("Artefatos salvos:")
print(f"  1. {model_path}")
print(f"  2. {le_path}")
print(f"  3. reports/figures/fig16_mlp_training_curves.png")
print(f"  4. reports/figures/fig17_mlp_confusion_matrix.png")